In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
%cd /content/drive/MyDrive/VKR_TECD/
os.makedirs('output', exist_ok=True)
print(os.getcwd())

/content/drive/MyDrive/VKR_TECD
/content/drive/MyDrive/VKR_TECD


In [6]:
import numpy as np
from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
import pickle
import time
import json

print("ШАГ 2: ОБУЧЕНИЕ SVD")

print("Загрузка матрицы...")
R = load_npz("output/train_matrix.npz")
print(f"Матрица: {R.shape[0]:,} x {R.shape[1]:,}")

k = 50
print(f"Параметры SVD: k = {k}")

print("Обучение модели...")
start_time = time.time()

svd = TruncatedSVD(n_components=k, random_state=42, algorithm='randomized', n_iter=5)
svd.fit(R)

train_time = time.time() - start_time
print(f"Обучение завершено за {train_time:.2f} секунд")

user_factors = svd.transform(R)
item_factors = svd.components_.T

print(f"Матрица пользователей P: {user_factors.shape}")
print(f"Матрица объектов Q: {item_factors.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Объясненная дисперсия: {explained_variance:.4%}")

with open("output/svd_model.pkl", "wb") as f:
    pickle.dump(svd, f)

np.save("output/user_factors.npy", user_factors)
np.save("output/item_factors.npy", item_factors)

params = {
    'model': 'SVD',
    'k': int(k),
    'train_time_sec': float(train_time),
    'explained_variance': float(explained_variance)
}
with open("output/svd_params.json", "w") as f:
    json.dump(params, f, indent=2)

print("Модель сохранена в output/")

ШАГ 2: ОБУЧЕНИЕ SVD
Загрузка матрицы...
Матрица: 1,382,085 x 760,794
Параметры SVD: k = 50
Обучение модели...
Обучение завершено за 107.31 секунд
Матрица пользователей P: (1382085, 50)
Матрица объектов Q: (760794, 50)
Объясненная дисперсия: 35.7211%
Модель сохранена в output/


In [7]:
!ls output/

item2idx.pkl	  statistics.json  train_matrix.npz  user_ids.npy
item_factors.npy  svd_model.pkl    user2idx.pkl
item_ids.npy	  svd_params.json  user_factors.npy
